In [1]:
from bs4 import BeautifulSoup
import requests
import re
import numpy as np
import pandas as pd
from urllib.parse import urljoin
from pathlib import Path

In [2]:
headers = {
    'User-agent' : 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'
}

# Setting the table with players

In [3]:
import numpy as np
import pandas as pd
from urllib.parse import urljoin

In [4]:
season_2025 = pd.DataFrame()

In [5]:
year = '2025'
url = f'https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1/plus/?saison_id={year}'

response = requests.get(url=url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
table_teams = soup.select_one('table[class="items"]')

In [6]:
teams = []

for a in soup.select("table.items tbody td.hauptlink a"):
    name = a.get_text(strip=True)
    href = urljoin(url, a["href"])
    teams.append((name, href))

teams = teams[:-5]

Cleaning the vector

In [7]:
teams = [i for i in teams if i[0] != '']
teams = teams[:20]

In [8]:
teams

[('Real Madrid',
  'https://www.transfermarkt.us/real-madrid/startseite/verein/418/saison_id/2025'),
 ('FC Barcelona',
  'https://www.transfermarkt.us/fc-barcelona/startseite/verein/131/saison_id/2025'),
 ('Atlético de Madrid',
  'https://www.transfermarkt.us/atletico-madrid/startseite/verein/13/saison_id/2025'),
 ('Athletic Bilbao',
  'https://www.transfermarkt.us/athletic-bilbao/startseite/verein/621/saison_id/2025'),
 ('Villarreal CF',
  'https://www.transfermarkt.us/fc-villarreal/startseite/verein/1050/saison_id/2025'),
 ('Real Sociedad',
  'https://www.transfermarkt.us/real-sociedad-san-sebastian/startseite/verein/681/saison_id/2025'),
 ('Real Betis Balompié',
  'https://www.transfermarkt.us/real-betis-sevilla/startseite/verein/150/saison_id/2025'),
 ('Valencia CF',
  'https://www.transfermarkt.us/fc-valencia/startseite/verein/1049/saison_id/2025'),
 ('Girona FC',
  'https://www.transfermarkt.us/fc-girona/startseite/verein/12321/saison_id/2025'),
 ('Celta de Vigo',
  'https://www.

In [9]:
players = []

for i in teams:
    url = i[1]
    response = requests.get(url = url, headers = headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    names = []
    for row in soup.select("table.items tbody tr"):
        cell = row.select_one("td.hauptlink a")
        if not cell:
            continue
        
        name = cell.get_text(strip=True)
        if name not in names:
            link = urljoin(url, cell["href"])
            players.append((i[0], name, link))
            names.append(name)

In [10]:
len(players)

500

## Generalizing for seasons from 17 to 24, so we get all the data we need

In [11]:
years = [str(2017 + i) for i in range(9)]
players = []


for year in years:
    url = f'https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1/plus/?saison_id={year}'
    response = requests.get(url=url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    table_teams = soup.select_one('table[class="items"]')  # table of teams per season

    teams = []

    for a in soup.select("table.items tbody td.hauptlink a"):
        name = a.get_text(strip=True)
        href = urljoin(url, a["href"])
        teams.append((name, href))
    
    teams = teams[:-5]  
    
    teams = [i for i in teams if i[0] != '']
    teams = teams[:20]  # here a vector of all teams cleaned
    
    for i in teams:
        url = i[1]
        response = requests.get(url = url, headers = headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        names = []
        for row in soup.select("table.items tbody tr"):
            cell = row.select_one("td.hauptlink a")
            if not cell:
                continue
            
            name = cell.get_text(strip=True)
            if name not in names:
                link = urljoin(url, cell["href"])
                player_id = link.split('/')[-1]
                players.append((year, i[0], player_id, name, link))
                names.append(name)

In [12]:
len(players)

6386

In [13]:
columns = ['year', 'club', 'player_id', 'player_name', 'link_html']
df = pd.DataFrame(players, columns = columns)
df

,year,club,player_id,player_name,link_html
0,2017,FC Barcelona,74857,Marc-André ter Stegen,https://www.transfermarkt.us/marc-andre-ter-st...
1,2017,FC Barcelona,146227,Jasper Cillessen,https://www.transfermarkt.us/jasper-cillessen/...
2,2017,FC Barcelona,142033,Adrián Ortolá,https://www.transfermarkt.us/adrian-ortola/pro...
3,2017,FC Barcelona,126540,Samuel Umtiti,https://www.transfermarkt.us/samuel-umtiti/pro...
4,2017,FC Barcelona,18944,Gerard Piqué,https://www.transfermarkt.us/gerard-pique/prof...
...,...,...,...,...,...
6381,2025,Real Oviedo,337284,Ovie Ejaria,https://www.transfermarkt.us/ovie-ejaria/profi...
6382,2025,Real Oviedo,620164,Haissem Hassan,https://www.transfermarkt.us/haissem-hassan/pr...
6383,2025,Real Oviedo,578419,Federico Viñas,https://www.transfermarkt.us/federico-vinas/pr...
6384,2025,Real Oviedo,743951,Thiago Borbas,https://www.transfermarkt.us/thiago-borbas/pro...


In [14]:
base_path = Path("/Users/luccagazotto/Documents/Personal/ML Projects/Soccer player/Soccer_player_market_value")
base_path.mkdir(parents=True, exist_ok=True)

In [15]:
df.to_parquet(base_path / "players_panel.parquet", index=False)

In [16]:
df = pd.read_parquet(base_path / "players_panel.parquet")
df.head()

,year,club,player_id,player_name,link_html
0,2017,FC Barcelona,74857,Marc-André ter Stegen,https://www.transfermarkt.us/marc-andre-ter-st...
1,2017,FC Barcelona,146227,Jasper Cillessen,https://www.transfermarkt.us/jasper-cillessen/...
2,2017,FC Barcelona,142033,Adrián Ortolá,https://www.transfermarkt.us/adrian-ortola/pro...
3,2017,FC Barcelona,126540,Samuel Umtiti,https://www.transfermarkt.us/samuel-umtiti/pro...
4,2017,FC Barcelona,18944,Gerard Piqué,https://www.transfermarkt.us/gerard-pique/prof...


In [17]:
def value_near_december(df, year):
    # df must have columns: date (datetime), market_value_eur, age
    try:
        dec = df[(df.date.dt.year == year) & (df.date.dt.month == 12)]
        if not dec.empty:
            return dec.sort_values('date').iloc[-1]
    
        nov = df[(df.date.dt.year == year) & (df.date.dt.month == 11)]
        if not nov.empty:
            return nov.sort_values('date').iloc[-1]
    
        jan = df[(df.date.dt.year == year + 1) & (df.date.dt.month == 1)]
        if not jan.empty:
            return jan.sort_values('date').iloc[0]
    
        return None
    except:
        return None

def value_near_december_value_age(df, year):
    row = value_near_december(df, year)
    if row is None:
        return None, None
    return int(row["market_value_eur"]), row["age"]

In [18]:
vector_age = []
vector_market_value = []
vector_nationality = []
vector_height = []
vector_foot = []
vector_position = []

for i in range(len(df)):
    url = df.iloc[i]['link_html']
    player_id = df.iloc[i]['player_id']
    response = requests.get(url=url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    year = int(df.iloc[i]['year'])
    
    vector_nationality.append(soup.select_one('span[itemprop="nationality"]').get_text(strip=True))  # nationality
    
    try:
        height = soup.select_one('span[itemprop="height"]').get_text(strip=True)[:4]
        height = float(height.replace(',','.'))
        vector_height.append(height)  # height
    except:
        vector_height.append(None)  # height
    
    try:
        foot = next(
        label.find_next_sibling('span', class_='info-table__content--bold').get_text(strip=True)
        for label in soup.select('span.info-table__content--regular')
        if label.get_text(strip=True) == 'Foot:'
        )
        vector_foot.append(foot)  # foot
    except:
        vector_foot.append(None)
    
    try:
        position = next(
        label.find_next_sibling('span', class_='info-table__content--bold').get_text(strip=True)
        for label in soup.select('span.info-table__content--regular')
        if label.get_text(strip=True) == 'Position:'
        )
        vector_position.append(position)
    except:
        vector_position.append(None)
    
    # market value & age
    
    api_link = f'https://tmapi-alpha.transfermarkt.technology/player/{player_id}/market-value-history'
    
    response_market_value = requests.get(url=api_link, headers=headers)
    
    json_root = response_market_value.json()      
    data = json_root.get('data', {})
    history = data.get('history', [])
    current = data.get('current', {})
    
    df_history = pd.DataFrame([
        {
            'player_id': h['playerId'],
            'club_id': h.get('clubId'),
            'date': pd.to_datetime(h['marketValue']['determined']),
            'age': h.get('age'),
            'market_value_eur': h['marketValue']['value']
        }
        for h in history
    ])
    
    market_value, age = value_near_december_value_age(df_history, year)

    vector_age.append(age)
    vector_market_value.append(market_value)
    

In [19]:
df['age'] = vector_age
df['height'] = vector_height
df['nationality'] = vector_nationality
df['foot'] = vector_foot
df['position'] = vector_position
df['market_value'] = vector_market_value

In [20]:
df.to_parquet(base_path / "players_panel.parquet", index=False)

In [21]:
df.head()

,year,club,player_id,player_name,link_html,age,height,nationality,foot,position,market_value
0,2017,FC Barcelona,74857,Marc-André ter Stegen,https://www.transfermarkt.us/marc-andre-ter-st...,25.0,1.87,Germany,right,Goalkeeper,40000000.0
1,2017,FC Barcelona,146227,Jasper Cillessen,https://www.transfermarkt.us/jasper-cillessen/...,28.0,1.87,Netherlands,right,Goalkeeper,9000000.0
2,2017,FC Barcelona,142033,Adrián Ortolá,https://www.transfermarkt.us/adrian-ortola/pro...,24.0,1.87,Spain,left,Goalkeeper,1500000.0
3,2017,FC Barcelona,126540,Samuel Umtiti,https://www.transfermarkt.us/samuel-umtiti/pro...,24.0,1.82,France,left,Defender - Centre-Back,40000000.0
4,2017,FC Barcelona,18944,Gerard Piqué,https://www.transfermarkt.us/gerard-pique/prof...,30.0,1.94,Spain,right,Defender - Centre-Back,40000000.0
